# Day 19: Target Scaling and Neural Network Improvement

## Objective
Investigate whether scaling target variables improves the learning
behaviour and prediction performance of a time-series neural network.

## Research Hypothesis
Scaling target variables may improve neural network optimization by
bringing output variables to comparable numerical ranges.

## Experiment Design
1. Load the processed wearable health dataset.
2. Generate time-series sequences.
3. Split data at participant level.
4. Scale input features using training data only.
5. Scale target variables using training data only.
6. Train a fresh neural network.
7. Inverse-transform predictions.
8. Compare performance against previous experiments and a mean baseline.

## Evaluation Metrics
- Mean Absolute Error (MAE)
- Steps deviation MAE
- Sleep deviation MAE

## Reproducibility
The experiment will use a fixed random seed and an independent
participant-level train-test split.

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Machine Learning
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# Deep Learning
import torch
import torch.nn as nn

# Reproducibility
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("All libraries imported successfully!")
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)

All libraries imported successfully!
PyTorch version: 2.14.1+cpu
NumPy version: 2.3.3


In [2]:
from pathlib import Path

# Project root directory
PROJECT_ROOT = Path.cwd().parent

# Dataset path
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "daily_features.csv"
)

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Unique participants:", df["Id"].nunique())
print("Columns:", df.columns.tolist())

Dataset loaded successfully!
Shape: (199, 21)
Unique participants: 20
Columns: ['Id', 'date', 'TotalSteps', 'Calories', 'VeryActiveMinutes', 'FairlyActiveMinutes', 'LightlyActiveMinutes', 'SedentaryMinutes', 'recorded_sleep_minutes', 'baseline_steps_7d', 'baseline_sleep_7d', 'steps_deviation_7d', 'steps_deviation_7d_pct', 'sleep_deviation_7d', 'sleep_deviation_7d_pct', 'steps_rolling_3d', 'sleep_rolling_3d', 'activity_ratio', 'sleep_ratio', 'day_number', 'is_baseline']


In [3]:
target_columns = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct"
]

print("Target variable statistics:\n")

print(
    df[target_columns].describe().round(2)
)

print("\nMissing values:")
print(df[target_columns].isnull().sum())

print("\nTarget correlations:")
print(df[target_columns].corr().round(3))

Target variable statistics:

       steps_deviation_7d_pct  sleep_deviation_7d_pct
count                  199.00                  199.00
mean                    -6.07                   -1.53
std                     39.98                   30.69
min                   -100.00                  -95.55
25%                    -24.39                  -12.53
50%                     -6.62                    1.06
75%                     15.45                   14.76
max                    121.62                   82.25

Missing values:
steps_deviation_7d_pct    0
sleep_deviation_7d_pct    0
dtype: int64

Target correlations:
                        steps_deviation_7d_pct  sleep_deviation_7d_pct
steps_deviation_7d_pct                   1.000                  -0.089
sleep_deviation_7d_pct                  -0.089                   1.000


In [4]:
FEATURE_COLUMNS = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct"
]

TARGET_COLUMNS = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct"
]

SEQUENCE_LENGTH = 3

X_sequences = []
y_targets = []
sequence_participants = []

sequence_data = df.copy()
sequence_data["date"] = pd.to_datetime(sequence_data["date"])

for participant_id, group in sequence_data.groupby("Id"):

    group = group.sort_values("date")

    features = group[FEATURE_COLUMNS].values
    targets = group[TARGET_COLUMNS].values

    for i in range(len(group) - SEQUENCE_LENGTH):

        X_sequences.append(
            features[i:i + SEQUENCE_LENGTH]
        )

        y_targets.append(
            targets[i + SEQUENCE_LENGTH]
        )

        sequence_participants.append(participant_id)

X_sequences = np.array(X_sequences)
y_targets = np.array(y_targets)
sequence_participants = np.array(sequence_participants)

print("Input sequences:", X_sequences.shape)
print("Target values:", y_targets.shape)
print("Participant IDs:", sequence_participants.shape)
print("Unique participants:", len(np.unique(sequence_participants)))

Input sequences: (140, 3, 2)
Target values: (140, 2)
Participant IDs: (140,)
Unique participants: 19


In [5]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_STATE
)

train_indices, test_indices = next(
    splitter.split(
        X_sequences,
        y_targets,
        groups=sequence_participants
    )
)

X_train = X_sequences[train_indices]
X_test = X_sequences[test_indices]

y_train = y_targets[train_indices]
y_test = y_targets[test_indices]

train_participants = set(
    sequence_participants[train_indices]
)

test_participants = set(
    sequence_participants[test_indices]
)

print("Training sequences:", X_train.shape)
print("Testing sequences:", X_test.shape)

print("Training participants:", len(train_participants))
print("Testing participants:", len(test_participants))

print("Participant overlap:",
      train_participants & test_participants)

Training sequences: (103, 3, 2)
Testing sequences: (37, 3, 2)
Training participants: 14
Testing participants: 5
Participant overlap: set()


In [6]:
input_scaler = StandardScaler()

X_train_scaled = input_scaler.fit_transform(
    X_train.reshape(-1, X_train.shape[-1])
).reshape(X_train.shape)

X_test_scaled = input_scaler.transform(
    X_test.reshape(-1, X_test.shape[-1])
).reshape(X_test.shape)

print("Scaled training input:", X_train_scaled.shape)
print("Scaled testing input:", X_test_scaled.shape)

Scaled training input: (103, 3, 2)
Scaled testing input: (37, 3, 2)


In [7]:
# Create a separate scaler for output variables
target_scaler = StandardScaler()

# Fit only on training targets
y_train_scaled = target_scaler.fit_transform(y_train)

# Apply the same transformation to test targets
y_test_scaled = target_scaler.transform(y_test)

print("Original training targets:")
print(y_train[:5])

print("\nScaled training targets:")
print(y_train_scaled[:5])

print("\nOriginal target shape:", y_train.shape)
print("Scaled target shape:", y_train_scaled.shape)

print("\nTraining target means:")
print(y_train_scaled.mean(axis=0).round(4))

print("\nTraining target standard deviations:")
print(y_train_scaled.std(axis=0).round(4))

Original training targets:
[[-23.58319796  50.96857512]
 [-27.41703411  35.90185105]
 [-59.09491761  34.09384417]
 [-18.02738454 -83.12526905]
 [-31.60826178  55.48859234]]

Scaled training targets:
[[-0.44757654  1.89331135]
 [-0.54584113  1.37257851]
 [-1.35777312  1.31009057]
 [-0.30517617 -2.7412109 ]
 [-0.65326598  2.0495312 ]]

Original target shape: (103, 2)
Scaled target shape: (103, 2)

Training target means:
[-0. -0.]

Training target standard deviations:
[1. 1.]


In [9]:
class HealthPatternModel(nn.Module):
    
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 2, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.network(x)


scaled_target_model = HealthPatternModel()

print(scaled_target_model)

HealthPatternModel(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=6, out_features=16, bias=True)
    (2): ReLU()
    (3): Linear(in_features=16, out_features=2, bias=True)
  )
)


In [10]:
# Convert input sequences
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

# Convert SCALED target variables
y_train_tensor = torch.tensor(
    y_train_scaled,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test_scaled,
    dtype=torch.float32
)

# Loss function and optimizer
loss_function = nn.MSELoss()

optimizer = torch.optim.Adam(
    scaled_target_model.parameters(),
    lr=0.001
)

# Training loop
epochs = 100

for epoch in range(epochs):

    scaled_target_model.train()

    predictions = scaled_target_model(X_train_tensor)

    loss = loss_function(
        predictions,
        y_train_tensor
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch + 1}, "
            f"Loss: {loss.item():.4f}"
        )

Epoch 20, Loss: 1.0573
Epoch 40, Loss: 1.0097
Epoch 60, Loss: 0.9766
Epoch 80, Loss: 0.9509
Epoch 100, Loss: 0.9280


In [11]:
# Switch model to evaluation mode
scaled_target_model.eval()

# Generate predictions without gradient calculation
with torch.no_grad():
    scaled_predictions = scaled_target_model(X_test_tensor)

# Convert PyTorch tensor to NumPy
scaled_predictions_np = scaled_predictions.cpu().numpy()

# Convert standardized predictions back to original scale
original_predictions = target_scaler.inverse_transform(
    scaled_predictions_np
)

print("Scaled predictions:")
print(scaled_predictions_np[:5])

print("\nOriginal-scale predictions:")
print(original_predictions[:5])

print("\nActual values:")
print(y_test[:5])

Scaled predictions:
[[-0.01428557  0.07415367]
 [-0.04705404 -0.00563297]
 [-0.33238953  0.03015286]
 [-0.18600066  0.19202977]
 [ 0.04517443  0.03324959]]

Original-scale predictions:
[[ -6.67816    -1.6663748]
 [ -7.9566364  -3.9748971]
 [-19.089127   -2.9394808]
 [-13.3777      1.7442161]
 [ -4.358302   -2.8498807]]

Actual values:
[[  1.62169872 -78.66817156]
 [ -7.51818651   8.23927765]
 [ -5.8668627   -3.87509406]
 [ -5.82077925 -12.30248307]
 [-13.60888313  55.64334086]]


In [12]:
# Overall MAE
overall_mae = mean_absolute_error(
    y_test,
    original_predictions
)

# Feature-wise MAE
steps_mae = mean_absolute_error(
    y_test[:, 0],
    original_predictions[:, 0]
)

sleep_mae = mean_absolute_error(
    y_test[:, 1],
    original_predictions[:, 1]
)

print(f"Overall MAE: {overall_mae:.4f}")
print(f"Steps deviation MAE: {steps_mae:.4f}")
print(f"Sleep deviation MAE: {sleep_mae:.4f}")

Overall MAE: 22.8521
Steps deviation MAE: 28.1761
Sleep deviation MAE: 17.5282


In [13]:
mean_baseline_prediction = np.tile(
    y_train.mean(axis=0),
    (len(y_test), 1)
)

baseline_overall_mae = mean_absolute_error(
    y_test,
    mean_baseline_prediction
)

baseline_steps_mae = mean_absolute_error(
    y_test[:, 0],
    mean_baseline_prediction[:, 0]
)

baseline_sleep_mae = mean_absolute_error(
    y_test[:, 1],
    mean_baseline_prediction[:, 1]
)

print("Mean baseline overall MAE:", baseline_overall_mae)
print("Mean baseline steps MAE:", baseline_steps_mae)
print("Mean baseline sleep MAE:", baseline_sleep_mae)

print("\nImprovement over baseline:")
print("Overall:", baseline_overall_mae - overall_mae)
print("Steps:", baseline_steps_mae - steps_mae)
print("Sleep:", baseline_sleep_mae - sleep_mae)

Mean baseline overall MAE: 22.49886893969092
Mean baseline steps MAE: 27.42983333788639
Mean baseline sleep MAE: 17.56790454149545

Improvement over baseline:
Overall: -0.3532433090828704
Steps: -0.7462204219535629
Sleep: 0.03973380378782565


## Day 19: Target Scaling Experiment

### Objective
Investigate whether standardizing target variables improves
neural network prediction performance.

### Methodology
- Generated 140 time-series sequences.
- Used 3 previous observations to predict the next observation.
- Applied participant-level train-test splitting.
- Training: 103 sequences from 14 participants.
- Testing: 37 sequences from 5 participants.
- Scaled input features and target variables independently.
- Fitted both scalers exclusively on training data.
- Trained a PyTorch neural network for 100 epochs.
- Applied inverse transformation before evaluation.

### Training Result
Final training MSE: 0.9280

### Evaluation Results

| Metric | Mean Baseline | Target-Scaled NN |
|---|---:|---:|
| Overall MAE | 22.4989 | 22.8521 |
| Steps MAE | 27.4298 | 28.1761 |
| Sleep MAE | 17.5679 | 17.5282 |

### Findings
Target scaling reduced the numerical scale of the training loss
but did not improve overall test prediction accuracy.

Steps prediction performed worse than the mean baseline.
Sleep prediction showed a negligible improvement.

### Conclusion
The experiment does not provide evidence that target scaling
alone improves generalization for this dataset and model architecture.

Further experiments should investigate:
- Target distribution and outliers.
- Model capacity and regularization.
- Temporal validation.
- Baseline prediction strategies.
- Additional participant data.

### Limitations
Only 37 test sequences from 5 participants were available.
The dataset is small and does not provide clinical ground truth.

The neural network remains an experimental model and is not
clinically validated.